# JSON (2026년 최신 권장 사용법)

`.json` 확장자를 가지는 파일을 `Document` 로 로드하는 방법을 살펴보겠습니다.

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전, `langchain_community`) | 현재 권장 |
|---|---|
| `JSONLoader(jq_schema=".[].phoneNumbers", text_content=False)` | 표준 라이브러리 `json` 으로 직접 값 추출 → `Document` (복잡한 쿼리가 필요하면 `jq` 패키지 직접 사용) |

`JSONLoader` 가 만들던 결과 형식(`page_content` = 값의 JSON 문자열, `metadata` = `source`, `seq_num`(1부터))을 그대로 재현합니다.

In [ ]:
# 설치
# !pip install -qU langchain-core
# !pip install -qU jq   # (선택) jq 문법을 쓰고 싶을 때

In [ ]:
import json
from pathlib import Path
from pprint import pprint

file_path = "data/people.json"
data = json.loads(Path(file_path).read_text(encoding="utf-8"))

pprint(data)

In [ ]:
type(data[0])

## JSON 값 → Document

JSON 데이터에서 각 사람의 `phoneNumbers` 필드 값을 추출하여 문서로 만든다고 가정해 봅시다.

### 방법 1) 순수 Python (의존성 없음)

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        # text_content=False 와 동일: dict/list 값을 JSON 문자열로 저장
        page_content=json.dumps(person.get("phoneNumbers"), ensure_ascii=False),
        metadata={"source": str(Path(file_path).resolve()), "seq_num": i},
    )
    for i, person in enumerate(data, start=1)
]

# 결과 출력
pprint(docs)

### 방법 2) `jq` 문법 사용

`jq` 쿼리에 익숙하다면 `jq` 파이썬 패키지를 직접 사용할 수 있습니다. (책의 `jq_schema` 와 같은 문법)

In [ ]:
import jq

values = jq.compile(".[].phoneNumbers").input_value(data).all()

docs = [
    Document(
        page_content=value if isinstance(value, str) else json.dumps(value, ensure_ascii=False),
        metadata={"source": str(Path(file_path).resolve()), "seq_num": i},
    )
    for i, value in enumerate(values, start=1)
]
pprint(docs)

### 응용: 본문 필드 + 메타데이터 필드 지정 (구 `content_key`, `metadata_func`)

레코드 하나를 문서 하나로 만들고, 일부 필드는 메타데이터로 옮기는 패턴입니다.

In [ ]:
def record_to_document(record: dict, seq_num: int) -> Document:
    content = {k: v for k, v in record.items() if k not in ("age",)}  # 본문에 쓸 필드
    return Document(
        page_content=json.dumps(content, ensure_ascii=False, indent=2),
        metadata={"source": file_path, "seq_num": seq_num, "age": record.get("age")},
    )


docs = [record_to_document(r, i) for i, r in enumerate(data, start=1)]
print(docs[0].page_content)
print(docs[0].metadata)

> JSON Lines(`.jsonl`) 파일은 한 줄씩 읽으면 됩니다.
>
> ```python
> with open("data.jsonl", encoding="utf-8") as f:
>     docs = [Document(page_content=json.loads(line)["text"], metadata={"line": i})
>             for i, line in enumerate(f, start=1) if line.strip()]
> ```